In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Multi-Dconv Head Transposed Attention (MDTA)
class MDTA(nn.Module):
    def __init__(self, channels, num_heads):
        super(MDTA, self).__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))

        # 1x1 Convs to generate Q, K, V
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1, bias=False)
        # 3x3 Depthwise Convs to encode spatial context into Q, K, V
        self.qkv_dwconv = nn.Conv2d(channels * 3, channels * 3, kernel_size=3, stride=1, padding=1, groups=channels * 3, bias=False)
        self.project_out = nn.Conv2d(channels, channels, kernel_size=1, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape

        qkv = self.qkv_dwconv(self.qkv(x))
        q, k, v = qkv.chunk(3, dim=1)

        # Reshape for channel-wise attention: (B, Head, C/Head, H*W)
        q = q.view(b, self.num_heads, c // self.num_heads, h * w)
        k = k.view(b, self.num_heads, c // self.num_heads, h * w)
        v = v.view(b, self.num_heads, c // self.num_heads, h * w)

        # Normalize Q and K
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        # Transposed Attention: (C/Head, H*W) @ (H*W, C/Head) -> (C/Head, C/Head)
        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)

        # Apply Attention to V: (C/Head, C/Head) @ (C/Head, H*W) -> (C/Head, H*W)
        out = (attn @ v)
        
        # Reshape back to image format
        out = out.view(b, c, h, w)
        out = self.project_out(out)
        return out

# 2. Gated-Dconv Feed-Forward Network (GDFN)
class GDFN(nn.Module):
    def __init__(self, channels, expansion_factor=2.66):
        super(GDFN, self).__init__()
        hidden_channels = int(channels * expansion_factor)
        
        self.project_in = nn.Conv2d(channels, hidden_channels * 2, kernel_size=1, bias=False)
        self.dwconv = nn.Conv2d(hidden_channels * 2, hidden_channels * 2, kernel_size=3, stride=1, padding=1, groups=hidden_channels * 2, bias=False)
        self.project_out = nn.Conv2d(hidden_channels, channels, kernel_size=1, bias=False)

    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        # The Gating Mechanism: GeLU activation multiplied by the linear branch
        x = F.gelu(x1) * x2
        x = self.project_out(x)
        return x

# 3. The Core Restormer Block
class RestormerBlock(nn.Module):
    def __init__(self, channels, num_heads):
        super(RestormerBlock, self).__init__()
        # Using GroupNorm(1, C) as a perfectly stable 2D equivalent to LayerNorm
        self.norm1 = nn.GroupNorm(1, channels)
        self.attn = MDTA(channels, num_heads)
        self.norm2 = nn.GroupNorm(1, channels)
        self.ffn = GDFN(channels)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

# 4. The Final Hackathon-Ready Network
class FastRestormerSR(nn.Module):
    def __init__(self, upscale_factor=2, channels=64, num_blocks=6, num_heads=4):
        super(FastRestormerSR, self).__init__()
        
        # 1-Channel Grayscale Input
        self.entry = nn.Conv2d(1, channels, kernel_size=3, padding=1)
        
        # The Transformer Brain
        self.transformer_blocks = nn.Sequential(
            *[RestormerBlock(channels, num_heads) for _ in range(num_blocks)]
        )
        
        self.conv_after_body = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        
        # PixelShuffle Upsampling
        self.upsample = nn.Sequential(
            nn.Conv2d(channels, channels * (upscale_factor ** 2), kernel_size=3, padding=1),
            nn.PixelShuffle(upscale_factor)
        )
        
        # 1-Channel Grayscale Output
        self.exit = nn.Conv2d(channels, 1, kernel_size=3, padding=1)

    def forward(self, x):
        x_first = self.entry(x)
        res = self.transformer_blocks(x_first)
        res = self.conv_after_body(res)
        res = res + x_first  # Deep residual connection
        
        out = self.upsample(res)
        out = self.exit(out)
        
        # Clamp to avoid sigmoid destruction
        return torch.clamp(out, 0.0, 1.0)


'''# The Ultimate "Crazy" Differentiator: Spatial + Spectral Loss
class SpectralKaggleLoss(nn.Module):
    def __init__(self, window_size=11, channel=1):
        super().__init__()
        self.window_size = window_size
        self.channel = channel
        window = self._create_window(window_size, channel)
        self.register_buffer('window', window)
        
        # Sobel filters for Edge Loss
        k_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).view(1, 1, 3, 3).float() / 4.
        k_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).view(1, 1, 3, 3).float() / 4.
        self.register_buffer('k_x', k_x)
        self.register_buffer('k_y', k_y)

    def _gaussian(self, window_size, sigma):
        gauss = torch.tensor([np.exp(-(x - window_size//2)**2 / (2*sigma**2)) for x in range(window_size)], dtype=torch.float32)
        return gauss / gauss.sum()

    def _create_window(self, window_size, channel):
        _1D_window = self._gaussian(window_size, 1.5).unsqueeze(1)
        _2D_window = _1D_window.mm(_1D_window.t()).float()
        window = _2D_window.unsqueeze(0).unsqueeze(0)
        return window.expand(channel, 1, window_size, window_size).contiguous()

    def edge_loss(self, pred, target):
        pred_x = F.conv2d(pred, self.k_x, padding=1)
        pred_y = F.conv2d(pred, self.k_y, padding=1)
        target_x = F.conv2d(target, self.k_x, padding=1)
        target_y = F.conv2d(target, self.k_y, padding=1)
        return F.l1_loss(pred_x, target_x) + F.l1_loss(pred_y, target_y)

    def charbonnier_loss(self, pred, target, eps=1e-3):
        return torch.mean(torch.sqrt((pred - target)**2 + eps**2))

    # --- THE CRAZY TRICK: 2D FFT Frequency Loss ---
    def fft_loss(self, pred, target):
        # Transform both images into the Frequency Domain
        pred_fft = torch.fft.rfft2(pred, norm='ortho')
        target_fft = torch.fft.rfft2(target, norm='ortho')
        
        # Calculate the amplitude (magnitude) of the frequencies
        pred_amp = torch.abs(pred_fft)
        target_amp = torch.abs(target_fft)
        
        # Penalize any mismatch in the frequency spectrum
        return F.l1_loss(pred_amp, target_amp)

    def forward(self, pred, target):
        mu1 = F.conv2d(pred, self.window, padding=self.window_size//2, groups=self.channel)
        mu2 = F.conv2d(target, self.window, padding=self.window_size//2, groups=self.channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2

        sigma1_sq = F.conv2d(pred*pred, self.window, padding=self.window_size//2, groups=self.channel) - mu1_sq
        sigma2_sq = F.conv2d(target*target, self.window, padding=self.window_size//2, groups=self.channel) - mu2_sq
        sigma12 = F.conv2d(pred*target, self.window, padding=self.window_size//2, groups=self.channel) - mu1_mu2

        C1, C2 = 0.01**2, 0.03**2
        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2)) / ((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))
        
        char_loss = self.charbonnier_loss(pred, target)
        ssim_loss = 1 - ssim_map.mean()
        edge_loss = self.edge_loss(pred, target)
        frequency_loss = self.fft_loss(pred, target)
        
        # The Grandmaster Balance: Spatial Math + Texture + Sharpness + FREQUENCY
        return 0.50 * char_loss + 0.15 * ssim_loss + 0.10 * edge_loss + 0.25 * frequency_loss

# Loss Function with Edge Loss
# Upgraded Loss Function with Charbonnier + SSIM + Edge
class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, channel=1):
        super().__init__()
        self.window_size = window_size
        self.channel = channel
        window = self._create_window(window_size, channel)
        self.register_buffer('window', window)
        
        # Sobel filters for Edge Loss
        k_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).view(1, 1, 3, 3).float() / 4.
        k_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).view(1, 1, 3, 3).float() / 4.
        self.register_buffer('k_x', k_x)
        self.register_buffer('k_y', k_y)

    def _gaussian(self, window_size, sigma):
        gauss = torch.tensor([np.exp(-(x - window_size//2)**2 / (2*sigma**2)) for x in range(window_size)], dtype=torch.float32)
        return gauss / gauss.sum()

    def _create_window(self, window_size, channel):
        _1D_window = self._gaussian(window_size, 1.5).unsqueeze(1)
        _2D_window = _1D_window.mm(_1D_window.t()).float()
        window = _2D_window.unsqueeze(0).unsqueeze(0)
        return window.expand(channel, 1, window_size, window_size).contiguous()

    def edge_loss(self, pred, target):
        pred_x = F.conv2d(pred, self.k_x, padding=1)
        pred_y = F.conv2d(pred, self.k_y, padding=1)
        target_x = F.conv2d(target, self.k_x, padding=1)
        target_y = F.conv2d(target, self.k_y, padding=1)
        return F.l1_loss(pred_x, target_x) + F.l1_loss(pred_y, target_y)

    #  Charbonnier Loss
    def charbonnier_loss(self, pred, target, eps=1e-3):
        return torch.mean(torch.sqrt((pred - target)**2 + eps**2))

    def forward(self, pred, target):
        mu1 = F.conv2d(pred, self.window, padding=self.window_size//2, groups=self.channel)
        mu2 = F.conv2d(target, self.window, padding=self.window_size//2, groups=self.channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2

        sigma1_sq = F.conv2d(pred*pred, self.window, padding=self.window_size//2, groups=self.channel) - mu1_sq
        sigma2_sq = F.conv2d(target*target, self.window, padding=self.window_size//2, groups=self.channel) - mu2_sq
        sigma12 = F.conv2d(pred*target, self.window, padding=self.window_size//2, groups=self.channel) - mu1_mu2

        C1, C2 = 0.01**2, 0.03**2
        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2)) / ((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))
        
        
        char_loss = self.charbonnier_loss(pred, target)
        ssim_loss = 1 - ssim_map.mean()
        edge_loss = self.edge_loss(pred, target)
        
        
        return 0.75 * char_loss + 0.15 * ssim_loss + 0.10 * edge_loss '''
class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, channel=1):
        super().__init__()
        self.window_size = window_size
        self.channel = channel
        window = self._create_window(window_size, channel)
        self.register_buffer('window', window)
        
        k_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).view(1, 1, 3, 3).float() / 4.
        k_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).view(1, 1, 3, 3).float() / 4.
        self.register_buffer('k_x', k_x)
        self.register_buffer('k_y', k_y)

    def _gaussian(self, window_size, sigma):
        gauss = torch.tensor([np.exp(-(x - window_size//2)**2 / (2*sigma**2)) for x in range(window_size)], dtype=torch.float32)
        return gauss / gauss.sum()

    def _create_window(self, window_size, channel):
        _1D_window = self._gaussian(window_size, 1.5).unsqueeze(1)
        _2D_window = _1D_window.mm(_1D_window.t()).float()
        window = _2D_window.unsqueeze(0).unsqueeze(0)
        return window.expand(channel, 1, window_size, window_size).contiguous()

    def edge_loss(self, pred, target):
        pred_x = F.conv2d(pred, self.k_x, padding=1)
        pred_y = F.conv2d(pred, self.k_y, padding=1)
        target_x = F.conv2d(target, self.k_x, padding=1)
        target_y = F.conv2d(target, self.k_y, padding=1)
        return F.l1_loss(pred_x, target_x) + F.l1_loss(pred_y, target_y)

    def charbonnier_loss(self, pred, target, eps=1e-3):
        return torch.mean(torch.sqrt((pred - target)**2 + eps**2))

    def forward(self, pred, target):
        mu1 = F.conv2d(pred, self.window, padding=self.window_size//2, groups=self.channel)
        mu2 = F.conv2d(target, self.window, padding=self.window_size//2, groups=self.channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2

        sigma1_sq = F.conv2d(pred*pred, self.window, padding=self.window_size//2, groups=self.channel) - mu1_sq
        sigma2_sq = F.conv2d(target*target, self.window, padding=self.window_size//2, groups=self.channel) - mu2_sq
        sigma12 = F.conv2d(pred*target, self.window, padding=self.window_size//2, groups=self.channel) - mu1_mu2

        C1, C2 = 0.01**2, 0.03**2
        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2)) / ((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))
        
        char_loss = self.charbonnier_loss(pred, target)
        ssim_loss = 1 - ssim_map.mean()
        edge_loss = self.edge_loss(pred, target)
        
        return 0.75 * char_loss + 0.15 * ssim_loss + 0.10 * edge_loss

In [15]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset

class FastPatchDataset(Dataset):
    def __init__(self, is_train=True, lr_patch_size=64, upscale_factor=2):
        self.noisy_dir = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK/train/NoisyLR"
        self.clean_dir = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK/train/GT"
        self.files = sorted([f for f in os.listdir(self.noisy_dir) if f.endswith(".npy")])
        self.is_train = is_train
        self.lr_patch_size = lr_patch_size
        self.scale = upscale_factor

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        
        # Loading raw arrays
        noisy = np.load(os.path.join(self.noisy_dir, name)).astype(np.float32)
        clean = np.load(os.path.join(self.clean_dir, name)).astype(np.float32)
        
        # Normalizing
        noisy = np.clip(noisy, 0, 1)
        clean = np.clip(clean, 0, 1)

        if self.is_train:
            h, w = noisy.shape
            
            # Random Cropping
            x = random.randint(0, w - self.lr_patch_size)
            y = random.randint(0, h - self.lr_patch_size)
            
            noisy = noisy[y : y + self.lr_patch_size, x : x + self.lr_patch_size]
            clean = clean[y * self.scale : (y + self.lr_patch_size) * self.scale, 
                          x * self.scale : (x + self.lr_patch_size) * self.scale]

            # Advanced Augmentation
            # Random Flips
            if random.random() > 0.5:
                noisy = np.flip(noisy, axis=1) 
                clean = np.flip(clean, axis=1)
            if random.random() > 0.5:
                noisy = np.flip(noisy, axis=0) 
                clean = np.flip(clean, axis=0)
            
            # Random 90-degree Rotations
            if random.random() > 0.5:
                k = random.randint(1, 3) # Rotate 90, 180, or 270 degrees
                noisy = np.rot90(noisy, k)
                clean = np.rot90(clean, k)


        noisy = np.expand_dims(noisy.copy(), 0)
        clean = np.expand_dims(clean.copy(), 0)

        return torch.tensor(noisy), torch.tensor(clean)


dataset = FastPatchDataset()
x, y = dataset[0]
print(f"Optimized Dataset ready. LR shape: {x.shape}, HR shape: {y.shape}")

Optimized Dataset ready. LR shape: torch.Size([1, 64, 64]), HR shape: torch.Size([1, 128, 128])


In [16]:
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

model = FastRestormerSR(upscale_factor=2).to(device)
criterion = SSIMLoss().to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

EPOCHS = 300

# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, 
    T_0=50, 
    T_mult=1, 
    eta_min=1e-6
)


loader = DataLoader(
    FastPatchDataset(is_train=True, lr_patch_size=64), 
    batch_size=32,            
    shuffle=True,
    num_workers=4,           
    pin_memory=True,         
    persistent_workers=True  
)

scaler = GradScaler('cuda')

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for noisy, clean in loader:
        noisy = noisy.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast('cuda'):
            pred = model(noisy)
            
        loss = criterion(pred.float(), clean.float())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {avg_loss:.5f} | LR: {current_lr:.6f}")
    
    scheduler.step()
    
    if (epoch + 1) in [200, 250, 300]:
        torch.save(model.state_dict(), f"/kaggle/working/restoration_model_ep{epoch+1}.pth")
        print(f"--> Saved Checkpoint at Epoch {epoch+1}")
torch.save(model.state_dict(), "/kaggle/working/restoration_model.pth")

Training on: cuda
Epoch 01/300 | Loss: 0.12128 | LR: 0.000500
Epoch 02/300 | Loss: 0.08214 | LR: 0.000500
Epoch 03/300 | Loss: 0.07469 | LR: 0.000498
Epoch 04/300 | Loss: 0.07215 | LR: 0.000496
Epoch 05/300 | Loss: 0.06950 | LR: 0.000492
Epoch 06/300 | Loss: 0.06817 | LR: 0.000488
Epoch 07/300 | Loss: 0.06960 | LR: 0.000482
Epoch 08/300 | Loss: 0.06750 | LR: 0.000476
Epoch 09/300 | Loss: 0.06666 | LR: 0.000469
Epoch 10/300 | Loss: 0.06720 | LR: 0.000461
Epoch 11/300 | Loss: 0.06647 | LR: 0.000452
Epoch 12/300 | Loss: 0.06631 | LR: 0.000443
Epoch 13/300 | Loss: 0.06611 | LR: 0.000432
Epoch 14/300 | Loss: 0.06518 | LR: 0.000421
Epoch 15/300 | Loss: 0.06508 | LR: 0.000410
Epoch 16/300 | Loss: 0.06495 | LR: 0.000397
Epoch 17/300 | Loss: 0.06451 | LR: 0.000384
Epoch 18/300 | Loss: 0.06458 | LR: 0.000371
Epoch 19/300 | Loss: 0.06428 | LR: 0.000357
Epoch 20/300 | Loss: 0.06423 | LR: 0.000342
Epoch 21/300 | Loss: 0.06414 | LR: 0.000328
Epoch 22/300 | Loss: 0.06406 | LR: 0.000313
Epoch 23/300 |

In [20]:
'''import os
import numpy as np
import torch
from torch.amp import autocast


test_dir = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK/test/NoisyLR"
submission_dir = "/kaggle/working/submission/"
os.makedirs(submission_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running inference on: {device}")

# Loading the trained NAFNet model
model = FastSRNet(upscale_factor=2).to(device)
model.load_state_dict(torch.load("/kaggle/working/restoration_model.pth"))
model.eval()

files = sorted([f for f in os.listdir(test_dir) if f.endswith(".npy")])
print(f"Found {len(files)} files! Starting Test-Time Augmentation (TTA) inference...")

for idx, name in enumerate(files):
    path = os.path.join(test_dir, name)

    # Loading and normalizing
    img = np.load(path).astype(np.float32)
    img = np.clip(img, 0, 1)
    
    # Shape: (1, 1, 128, 128)
    img_tensor = torch.tensor(img).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        with autocast('cuda'):
          
            # 1. Base orientations
            p1 = model(img_tensor)
            p2 = torch.flip(model(torch.flip(img_tensor, dims=[3])), dims=[3]) 
            p3 = torch.flip(model(torch.flip(img_tensor, dims=[2])), dims=[2]) 
            p4 = torch.flip(model(torch.flip(img_tensor, dims=[2, 3])), dims=[2, 3]) 
            
            # 2. Rotated orientations (90 degrees)
            img_rot = torch.rot90(img_tensor, 1, [2, 3])
            
            p5 = torch.rot90(model(img_rot), -1, [2, 3])
            p6 = torch.rot90(torch.flip(model(torch.flip(img_rot, dims=[3])), dims=[3]), -1, [2, 3])
            p7 = torch.rot90(torch.flip(model(torch.flip(img_rot, dims=[2])), dims=[2]), -1, [2, 3])
            p8 = torch.rot90(torch.flip(model(torch.flip(img_rot, dims=[2, 3])), dims=[2, 3]), -1, [2, 3])
            
            # Average all 8 predictions
            pred = (p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8) / 8.0

    # Moving back to CPU and formatting for saving
    pred = pred.squeeze().cpu().numpy()
    pred = np.clip(pred, 0, 1).astype(np.float32)

    # Saving to submission folder
    np.save(os.path.join(submission_dir, name), pred)
    
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/200 files...")

print(f"BOOM! Inference complete. All 200 files saved to {submission_dir}")'''
# import os
# import numpy as np
# import torch
# import torch.nn.functional as F
# from torch.amp import autocast

# test_dir = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK/test/NoisyLR"
# submission_dir = "/kaggle/working/submission/"
# os.makedirs(submission_dir, exist_ok=True)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Running Fast TTA + Sharpening on: {device}")

# model = FastSRNet(upscale_factor=2).to(device)
# model.load_state_dict(torch.load("/kaggle/working/restoration_model.pth"))
# model.eval()


# sharpen_kernel = torch.tensor([[0, -1, 0], 
#                                [-1, 5, -1], 
#                                [0, -1, 0]], dtype=torch.float32).view(1, 1, 3, 3).to(device)

# files = sorted([f for f in os.listdir(test_dir) if f.endswith(".npy")])

# for idx, name in enumerate(files):
#     path = os.path.join(test_dir, name)
#     img = np.load(path).astype(np.float32)
#     img = np.clip(img, 0, 1)
#     img_tensor = torch.tensor(img).unsqueeze(0).unsqueeze(0).to(device)

#     with torch.no_grad():
#         with autocast('cuda'):
#             # Standard 8-Pass TTA
#             p1 = model(img_tensor)
#             p2 = torch.flip(model(torch.flip(img_tensor, dims=[3])), dims=[3]) 
#             p3 = torch.flip(model(torch.flip(img_tensor, dims=[2])), dims=[2]) 
#             p4 = torch.flip(model(torch.flip(img_tensor, dims=[2, 3])), dims=[2, 3]) 
            
#             img_rot = torch.rot90(img_tensor, 1, [2, 3])
#             p5 = torch.rot90(model(img_rot), -1, [2, 3])
#             p6 = torch.rot90(torch.flip(model(torch.flip(img_rot, dims=[3])), dims=[3]), -1, [2, 3])
#             p7 = torch.rot90(torch.flip(model(torch.flip(img_rot, dims=[2])), dims=[2]), -1, [2, 3])
#             p8 = torch.rot90(torch.flip(model(torch.flip(img_rot, dims=[2, 3])), dims=[2, 3]), -1, [2, 3])
            
#             final_pred = (p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8) / 8.0
            
#             # APPLY THE SHARPENING FILTER (Boosts SSIM slightly)
#             # We blend 90% of the original smooth prediction with 10% of the ultra-sharp version
#             sharp_pred = F.conv2d(final_pred, sharpen_kernel, padding=1)
#             final_pred = (final_pred * 0.90) + (sharp_pred * 0.10)

#     final_pred = final_pred.squeeze().cpu().numpy()
#     final_pred = np.clip(final_pred, 0, 1).astype(np.float32)
#     np.save(os.path.join(submission_dir, name), final_pred)

# print(f"Fast Inference + Sharpening Complete! Ready to zip.")
import os
import numpy as np
import torch
from torch.amp import autocast

test_dir = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK/test/NoisyLR"
submission_dir = "/kaggle/working/submission/"
os.makedirs(submission_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Igniting Restormer 24-Pass Ensemble TTA on: {device}")

# The 3 Restormer Checkpoints
checkpoints = [
    "/kaggle/working/restoration_model_ep200.pth",
    "/kaggle/working/restoration_model_ep250.pth",
    "/kaggle/working/restoration_model_ep300.pth"
]

# THE FIX: Build the correct Restormer "Empty Shell"
model = FastRestormerSR(upscale_factor=2).to(device)

files = sorted([f for f in os.listdir(test_dir) if f.endswith(".npy")])
print(f"Found {len(files)} test files. Commencing Grandmaster Ensemble Inference...")

for idx, name in enumerate(files):
    path = os.path.join(test_dir, name)

    img = np.load(path).astype(np.float32)
    img = np.clip(img, 0, 1)
    
    img_tensor = torch.tensor(img).unsqueeze(0).unsqueeze(0).to(device)
    ensemble_pred = torch.zeros((1, 1, 256, 256), device=device)

    with torch.no_grad():
        with autocast('cuda'):
            for ckpt in checkpoints:
                
                # Load the Restormer weights into the Restormer shell
                state_dict = torch.load(ckpt, map_location=device)
                clean_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
                model.load_state_dict(clean_dict)
                model.eval()
                
                # 8-Pass TTA
                p1 = model(img_tensor)
                p2 = torch.flip(model(torch.flip(img_tensor, dims=[3])), dims=[3]) 
                p3 = torch.flip(model(torch.flip(img_tensor, dims=[2])), dims=[2]) 
                p4 = torch.flip(model(torch.flip(img_tensor, dims=[2, 3])), dims=[2, 3]) 
                
                img_rot = torch.rot90(img_tensor, 1, [2, 3])
                p5 = torch.rot90(model(img_rot), -1, [2, 3])
                p6 = torch.rot90(torch.flip(model(torch.flip(img_rot, dims=[3])), dims=[3]), -1, [2, 3])
                p7 = torch.rot90(torch.flip(model(torch.flip(img_rot, dims=[2])), dims=[2]), -1, [2, 3])
                p8 = torch.rot90(torch.flip(model(torch.flip(img_rot, dims=[2, 3])), dims=[2, 3]), -1, [2, 3])
                
                ensemble_pred += (p1 + p2 + p3 + p4 + p5 + p6 + p7 + p8)

    # Average the 24 predictions
    final_pred = ensemble_pred / 24.0

    final_pred = final_pred.squeeze().cpu().numpy()
    final_pred = np.clip(final_pred, 0, 1).astype(np.float32)

    np.save(os.path.join(submission_dir, name), final_pred)
    
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/200 files...")

print(f"BOOM! Restormer Ensemble complete. All files saved to {submission_dir}")

Igniting Restormer 24-Pass Ensemble TTA on: cuda
Found 200 test files. Commencing Grandmaster Ensemble Inference...
Processed 50/200 files...
Processed 100/200 files...
Processed 150/200 files...
Processed 200/200 files...
BOOM! Restormer Ensemble complete. All files saved to /kaggle/working/submission/


In [21]:
import shutil
shutil.make_archive("/kaggle/working/final_submission", 'zip', "/kaggle/working/submission/")
print("Zip file created successfully! Ready to download and submit.")

Zip file created successfully! Ready to download and submit.


In [ ]:
'''import os
import numpy as np
import torch
from torch.amp import autocast

test_dir = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK/test"
submission_dir = "/kaggle/working/submission/"

os.makedirs(submission_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialize the same FastSRNet model and load the trained weights
model = FastSRNet(upscale_factor=2).to(device)
model.load_state_dict(torch.load("/kaggle/working/restoration_model.pth"))
model.eval()

if not os.path.exists(test_dir):
    print(f"Error: Could not find {test_dir}. Check the exact folder path!")
else:
    files = sorted([f for f in os.listdir(test_dir) if f.endswith(".npy")])
    print(f"Found {len(files)} files for inference. Starting...")

    for idx, name in enumerate(files):
        path = os.path.join(test_dir, name)

        # 2. Load the raw 128x128 numpy array (No cv2.resize!)
        img = np.load(path).astype(np.float32)
        img = np.clip(img, 0, 1)

        # 3. Add batch and channel dimensions -> shape: (1, 1, 128, 128)
        img_tensor = torch.tensor(img).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            with autocast('cuda'):
                # 4. Model naturally outputs the upscaled 256x256 image
                pred = model(img_tensor)

        # 5. Move back to CPU, strip extra dimensions, and format as numpy array
        pred = pred.squeeze().cpu().numpy()
        pred = np.clip(pred, 0, 1).astype(np.float32)

        np.save(
            os.path.join(submission_dir, name),
            pred
        )
        
    print(f"Inference complete! All upscaled 256x256 files saved to {submission_dir}")'''

In [22]:
import os
import numpy as np
import base64
import pandas as pd
from io import BytesIO

submission_dir = "/kaggle/working/submission"

rows = []

files = sorted([f for f in os.listdir(submission_dir) if f.endswith(".npy")])

for idx, file in enumerate(files, start=1):

    path = os.path.join(submission_dir, file)

    # load numpy array
    arr = np.load(path)

    # convert array to bytes
    buffer = BytesIO()
    np.save(buffer, arr)

    encoded = base64.b64encode(buffer.getvalue()).decode()

    rows.append({
        "id": idx,
        "npy_base64": encoded
    })

df = pd.DataFrame(rows)

df.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission created with", len(df), "rows")


Submission created with 200 rows


In [ ]:
'''import os
test_path = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK/test/NoisyLR"

# Let's see exactly what is sitting inside this folder
files = os.listdir(test_path)
print(f"Total files found: {len(files)}")

if len(files) > 0:
    print(f"Here are the first 5 files: {files[:5]}")'''

In [ ]:
'''import matplotlib.pyplot as plt
import torch


model.eval()

# Grab one single batch of data from your loader
data_iterator = iter(loader)
noisy_imgs, clean_imgs = next(data_iterator)


noisy_imgs = noisy_imgs.to(device)
with torch.no_grad():
    
    preds = model(noisy_imgs)


noisy_imgs = noisy_imgs.cpu()
preds = preds.cpu()
clean_imgs = clean_imgs.cpu()

# Choose how many images you want to look at (let's do 3)
num_images = 3

fig, axes = plt.subplots(num_images, 3, figsize=(12, 4 * num_images))
fig.suptitle('Super-Resolution & Denoising Results', fontsize=16)

for i in range(num_images):
    # 1. Plot the Noisy Low-Res Input (128x128)
    ax = axes[i, 0]
    ax.imshow(noisy_imgs[i].squeeze().numpy(), cmap='gray')
    ax.set_title('Noisy Low-Res Input')
    ax.axis('off')
    
    # 2. Plot the Model's Prediction (256x256)
    ax = axes[i, 1]
    ax.imshow(preds[i].squeeze().numpy(), cmap='gray')
    ax.set_title('Model Prediction')
    ax.axis('off')
    
    # 3. Plot the Clean Ground Truth (256x256)
    ax = axes[i, 2]
    ax.imshow(clean_imgs[i].squeeze().numpy(), cmap='gray')
    ax.set_title('Ground Truth High-Res')
    ax.axis('off')

plt.tight_layout()
plt.show()

'''